# 🔮 FORESIGHT — 04: Baseline & Classical Time-Series Forecasting

This notebook benchmarks baseline forecasting heuristics against classical statistical time series models:
- **Naive Forecaster** (Persistence)
- **Seasonal Naive Forecaster** (7-day weekly recurrence)
- **Moving Average Forecaster** (7-day rolling mean)
- **Holt-Winters Exponential Smoothing** (Additive level, trend & seasonal components)

In [ ]:
import pandas as pd
import numpy as np
from foresight.data.loader import load_processed_sales
from foresight.features.pipeline import FeatureEngineeringPipeline
from foresight.forecasting.baselines import (
    NaiveForecaster,
    SeasonalNaiveForecaster,
    MovingAverageForecaster,
    ExponentialSmoothingForecaster,
)
from foresight.evaluation.metrics import evaluate_predictions

# 1. Load Processed Dataset
df = pd.read_parquet('data/processed/features_engineered.parquet')
print(f"Loaded feature matrix: {len(df):,} rows")

## 2. Train-Validation Split (Temporal Cutoff)
Hold out the final 30 days of data for out-of-sample validation.

In [ ]:
dates = pd.to_datetime(df["date"])
val_start = dates.max() - pd.Timedelta(days=30)

train_df = df[dates < val_start]
val_df = df[dates >= val_start]

print(f"Train observations: {len(train_df):,} | Validation observations: {len(val_df):,}")

## 3. Fit and Evaluate Baseline Suite

In [ ]:
baselines = [
    NaiveForecaster(),
    SeasonalNaiveForecaster(season_length=7),
    MovingAverageForecaster(window=7),
    ExponentialSmoothingForecaster(season_length=7),
]

pipeline = FeatureEngineeringPipeline()
feat_cols = pipeline.get_feature_names(df)

y_train = train_df["quantity"].values
y_val = val_df["quantity"].values

print("=== BASELINE PERFORMANCE LEADERBOARD ===")
print(f"{'Model':<35} | {'WAPE':<8} | {'RMSE':<8} | {'MAE':<8} | {'sMAPE':<8}")
print("-" * 75)

for model in baselines:
    model.fit(train_df[feat_cols], y_train)
    preds = model.predict(val_df[feat_cols])
    scores = evaluate_predictions(y_val, preds, y_train=y_train)
    print(f"{model.name:<35} | {scores.wape:<8.4f} | {scores.rmse:<8.2f} | {scores.mae:<8.2f} | {scores.smape:<8.2f}%")